# Granite 4.1 — Document Q&A (RAG, MLX)

## Imports

In [1]:
from pprint import pprint

import mlx_lm

print("mlx-lm:", mlx_lm.__version__)

mlx-lm: 0.31.3


## Load Model and Tokenizer

In [2]:
MODEL_ID = "mlx-community/granite-4.1-3b-mxfp8"

model, tokenizer = mlx_lm.load(MODEL_ID)

print(f"Architecture: {model.model_type}")
print(f"Parameters: {mlx_lm.utils.get_total_parameters(model):,}")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Architecture: granite
Parameters: 3,402,836,480


## Define Documents

In [3]:
documents = [
    {
        "doc_id": 0,
        "title": "Mount Kilimanjaro",
        "text": (
            "Mount Kilimanjaro is a dormant volcano in Tanzania. It is the highest "
            "mountain in Africa, with a summit elevation of 5,895 metres (19,341 ft) "
            "above sea level."
        ),
    },
    {
        "doc_id": 1,
        "title": "Denali",
        "text": (
            "Denali, also known as Mount McKinley, is the highest mountain peak in "
            "North America, located in Alaska, with a summit elevation of 6,190 "
            "metres (20,310 ft)."
        ),
    },
    {
        "doc_id": 2,
        "title": "Mount Everest",
        "text": (
            "Mount Everest, located in the Mahalangur Himal sub-range of the "
            "Himalayas, is Earth's highest mountain above sea level, with a summit "
            "elevation of 8,849 metres (29,032 ft)."
        ),
    },
]

pprint(documents, sort_dicts=False, width=120)

[{'doc_id': 0,
  'title': 'Mount Kilimanjaro',
  'text': 'Mount Kilimanjaro is a dormant volcano in Tanzania. It is the highest mountain in Africa, with a summit '
          'elevation of 5,895 metres (19,341 ft) above sea level.'},
 {'doc_id': 1,
  'title': 'Denali',
  'text': 'Denali, also known as Mount McKinley, is the highest mountain peak in North America, located in Alaska, '
          'with a summit elevation of 6,190 metres (20,310 ft).'},
 {'doc_id': 2,
  'title': 'Mount Everest',
  'text': "Mount Everest, located in the Mahalangur Himal sub-range of the Himalayas, is Earth's highest mountain "
          'above sea level, with a summit elevation of 8,849 metres (29,032 ft).'}]


## Single Document Answer

In [4]:
messages = [
    {"role": "user", "content": "How tall is Mount Kilimanjaro?"},
]

chat = tokenizer.apply_chat_template(
    messages,
    documents=documents,
    tokenize=False,
    add_generation_prompt=True,
)

print(chat)

<|start_of_role|>system<|end_of_role|>You are a helpful assistant with access to the following documents. You may use one or more documents to assist with the user query.

You are given a list of documents within <documents></documents> XML tags:
<documents>
{"doc_id": 0, "title": "Mount Kilimanjaro", "text": "Mount Kilimanjaro is a dormant volcano in Tanzania. It is the highest mountain in Africa, with a summit elevation of 5,895 metres (19,341 ft) above sea level."}
{"doc_id": 1, "title": "Denali", "text": "Denali, also known as Mount McKinley, is the highest mountain peak in North America, located in Alaska, with a summit elevation of 6,190 metres (20,310 ft)."}
{"doc_id": 2, "title": "Mount Everest", "text": "Mount Everest, located in the Mahalangur Himal sub-range of the Himalayas, is Earth's highest mountain above sea level, with a summit elevation of 8,849 metres (29,032 ft)."}
</documents>

Write the response to the user's input by strictly aligning with the facts in the provid

In [5]:
response = mlx_lm.generate(model, tokenizer, chat, max_tokens=150)

print(response)

Mount Kilimanjaro is 5,895 metres (19,341 ft) tall.


## Multiple Document Synthesis

In [6]:
messages = [
    {"role": "user", "content": "Which is taller, Kilimanjaro or Denali?"},
]

chat = tokenizer.apply_chat_template(
    messages,
    documents=documents,
    tokenize=False,
    add_generation_prompt=True,
)

response = mlx_lm.generate(model, tokenizer, chat, max_tokens=150)

print(response)

Denali is taller than Kilimanjaro. Denali (Mount McKinley) has a summit elevation of 6,190 meters, while Mount Kilimanjaro stands at 5,895 meters.


## Unanswerable Query

In [7]:
messages = [
    {"role": "user", "content": "What is the population of Tanzania?"},
]

chat = tokenizer.apply_chat_template(
    messages,
    documents=documents,
    tokenize=False,
    add_generation_prompt=True,
)

response = mlx_lm.generate(model, tokenizer, chat, max_tokens=150)

print(response)

The question cannot be answered based on the available data. The provided documents contain information about Mount Kilimanjaro, Denali, and Mount Everest, but they do not include any data regarding the population of Tanzania.


## Multi-Turn with Documents

In [8]:
messages = [
    {"role": "user", "content": "How tall is Mount Kilimanjaro?"},
]

chat = tokenizer.apply_chat_template(
    messages,
    documents=documents,
    tokenize=False,
    add_generation_prompt=True,
)

response = mlx_lm.generate(model, tokenizer, chat, max_tokens=150)

print(response)

Mount Kilimanjaro is 5,895 metres (19,341 ft) tall.


In [9]:
messages.append({"role": "assistant", "content": response})
pprint(messages, sort_dicts=False, width=120)

[{'role': 'user', 'content': 'How tall is Mount Kilimanjaro?'},
 {'role': 'assistant', 'content': 'Mount Kilimanjaro is 5,895 metres (19,341 ft) tall.'}]


In [10]:
prompt = "How does that compare to Everest?"

messages.append({"role": "user", "content": prompt})
pprint(messages, sort_dicts=False, width=120)

[{'role': 'user', 'content': 'How tall is Mount Kilimanjaro?'},
 {'role': 'assistant', 'content': 'Mount Kilimanjaro is 5,895 metres (19,341 ft) tall.'},
 {'role': 'user', 'content': 'How does that compare to Everest?'}]


In [11]:
chat = tokenizer.apply_chat_template(
    messages,
    documents=documents,
    tokenize=False,
    add_generation_prompt=True,
)

response = mlx_lm.generate(model, tokenizer, chat, max_tokens=150)

print(response)

Mount Everest is 8,849 metres (29,032 ft) tall, which is significantly higher than Mount Kilimanjaro's 5,895 metres.
